# Build Final Merged BERTopic Artefacts

This notebook is the place where you create the final merged topic run. When you intentionally rerun it, it should overwrite the working merged artefacts, save the merged model, and freeze that rerun as the new final thesis run.

Until you rerun this notebook, the downstream analysis notebook should treat the current cached run as non-final.


In [ ]:
import json
import os
import sys
from pathlib import Path

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.7
SAVE_MERGED_MODEL = True
FINAL_RUN_LABEL = "final_merged_topics_v2"
AUTHORITATIVE_FINAL_RUN = True
FREEZE_THIS_RUN = True
ALLOW_OVERWRITE_FROZEN_RUN = False


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")
os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / ".mplconfig")

MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "1a_BERTopic" / "local_outputs",
    PROJECT_ROOT / "1a_BERTopic" / "outputs",
    PROJECT_ROOT / "BERTopic" / "outputs",
]
MERGED_SAVE_DIR = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_all_outlets_model"
RICH_EXPORT_PATH = PROJECT_ROOT / "data" / "processed" / "df_combined_with_merged_topics.csv"
TOPIC_EXPORT_PATH = PROJECT_ROOT / "data" / "processed" / "df_combined_with_topic.csv"
MERGED_ARTICLES_CACHE_PATH = PROJECT_ROOT / "data" / "processed" / "merged_articles_with_umap.csv"
MERGED_TOPIC_INFO_CACHE_PATH = PROJECT_ROOT / "data" / "processed" / "merged_topic_info_display.csv"
MERGED_METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "merged_analysis_metadata.json"

In [ ]:
import importlib

import pandas as pd
from IPython.display import display
from bertopic import BERTopic

import merged_outlets_analysis as moa
moa = importlib.reload(moa)

OUTLET_SPECS = moa.OUTLET_SPECS
REQUIRED_COMBINED_COLUMNS = moa.REQUIRED_COMBINED_COLUMNS
build_df_combined_with_topic = moa.build_df_combined_with_topic
build_merged_article_frame = moa.build_merged_article_frame
combine_preparation_audits = moa.combine_preparation_audits
combine_prepared_documents = moa.combine_prepared_documents
export_canonical_corpus_with_merged_topics = moa.export_canonical_corpus_with_merged_topics
export_df_combined_with_topic = moa.export_df_combined_with_topic
export_merged_analysis_cache = moa.export_merged_analysis_cache
freeze_merged_run_snapshot = moa.freeze_merged_run_snapshot
load_all_prepared_documents_with_audits = moa.load_all_prepared_documents_with_audits
load_canonical_combined_df = moa.load_canonical_combined_df
load_saved_merged_model = moa.load_saved_merged_model
resolve_model_paths = moa.resolve_model_paths
save_merged_model = moa.save_merged_model

In [ ]:
MODEL_PATHS = resolve_model_paths(MODEL_DIR_CANDIDATES)
SOURCE_MODEL_METADATA = {key: str(path.relative_to(PROJECT_ROOT)) for key, path in MODEL_PATHS.items()}

loaded_models = {
    key: BERTopic.load(model_path, embedding_model=EMBEDDING_MODEL)
    for key, model_path in MODEL_PATHS.items()
}

pd.DataFrame(
    {
        "Outlet": [OUTLET_SPECS[key].label for key in loaded_models],
        "Model_Path": [str(MODEL_PATHS[key]) for key in loaded_models],
    }
)

In [ ]:
if not SAVE_MERGED_MODEL:
    raise ValueError("The authoritative final merged run must save the merged model. Keep SAVE_MERGED_MODEL = True.")

models_to_merge = [
    loaded_models["tagesschau"],
    loaded_models["rt"],
    loaded_models["antispiegel"],
    loaded_models["tichys"],
    loaded_models["nius"],
    loaded_models["compact"],
    loaded_models["deutschlandkurier"],
]

candidate_merged_model = BERTopic.merge_models(
    models_to_merge,
    min_similarity=MIN_SIMILARITY,
    embedding_model=EMBEDDING_MODEL,
)

merged_model_path = save_merged_model(
    candidate_merged_model,
    PROJECT_ROOT,
    output_dir=MERGED_SAVE_DIR,
    embedding_model=EMBEDDING_MODEL,
)
merged_model, loaded_merged_model_path = load_saved_merged_model(
    PROJECT_ROOT,
    embedding_model=EMBEDDING_MODEL,
    model_dir=merged_model_path,
)
print(f"Saved and reloaded authoritative merged model from: {loaded_merged_model_path}")
print("Final run label:", FINAL_RUN_LABEL)
print("Merged model topic rows (raw BERTopic metadata):", len(merged_model.get_topic_info()))

In [ ]:
prepared_by_outlet, audit_by_outlet = load_all_prepared_documents_with_audits(PROJECT_ROOT)
combined_prepared = combine_prepared_documents(prepared_by_outlet)
combined_audit = combine_preparation_audits(audit_by_outlet)
canonical_df = load_canonical_combined_df(PROJECT_ROOT)

prepared_summary = pd.DataFrame(
    [
        {
            "Outlet": OUTLET_SPECS[key].label,
            "Prepared_Documents": len(prepared_by_outlet[key]),
        }
        for key in OUTLET_SPECS
    ]
)

display(prepared_summary)
print("Canonical rows:", len(canonical_df))
print("Prepared rows for merged transform:", len(combined_prepared))
print("Excluded before merged transform:", len(canonical_df) - len(combined_prepared))


In [ ]:
merged_articles, merged_topic_info_display, merged_umap_model = build_merged_article_frame(
    merged_model,
    combined_prepared,
)

if int(merged_topic_info_display["Count"].sum()) != len(merged_articles):
    raise ValueError("Merged topic-info counts no longer match the transformed article-level assignments.")

cache_articles_path, cache_topic_info_path, cache_metadata_path = export_merged_analysis_cache(
    PROJECT_ROOT,
    merged_articles,
    merged_topic_info_display,
    articles_output_path=MERGED_ARTICLES_CACHE_PATH,
    topic_info_output_path=MERGED_TOPIC_INFO_CACHE_PATH,
    metadata_output_path=MERGED_METADATA_PATH,
    merged_model_path=loaded_merged_model_path,
    metadata_overrides={
        "run_label": FINAL_RUN_LABEL,
        "authoritative_final_run": AUTHORITATIVE_FINAL_RUN,
        "min_similarity": MIN_SIMILARITY,
        "embedding_model": EMBEDDING_MODEL,
        "source_model_paths": SOURCE_MODEL_METADATA,
        "frozen_snapshot_dir": None,
    },
)
rich_export_path = export_canonical_corpus_with_merged_topics(
    PROJECT_ROOT,
    merged_articles,
    RICH_EXPORT_PATH,
    preparation_audit=combined_audit,
)
topic_export_path = export_df_combined_with_topic(
    PROJECT_ROOT,
    merged_articles,
    TOPIC_EXPORT_PATH,
    preparation_audit=combined_audit,
)

frozen_snapshot_dir = None
if FREEZE_THIS_RUN:
    frozen_snapshot_dir = freeze_merged_run_snapshot(
        PROJECT_ROOT,
        FINAL_RUN_LABEL,
        merged_model_path=loaded_merged_model_path,
        articles_path=cache_articles_path,
        topic_info_path=cache_topic_info_path,
        thesis_topic_export_path=topic_export_path,
        rich_topic_export_path=rich_export_path,
        metadata_path=cache_metadata_path,
        overwrite=ALLOW_OVERWRITE_FROZEN_RUN,
    )
    metadata = json.loads(cache_metadata_path.read_text())
    metadata["frozen_snapshot_dir"] = str(frozen_snapshot_dir.relative_to(PROJECT_ROOT))
    cache_metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
    (frozen_snapshot_dir / cache_metadata_path.name).write_text(json.dumps(metadata, indent=2) + "\n")

thesis_topic_df = build_df_combined_with_topic(
    PROJECT_ROOT,
    merged_articles,
    preparation_audit=combined_audit,
)
if not thesis_topic_df.loc[:, list(REQUIRED_COMBINED_COLUMNS)].equals(
    canonical_df.loc[:, list(REQUIRED_COMBINED_COLUMNS)]
):
    raise ValueError("The article-level Topic export no longer matches canonical df_combined columns.")

display(merged_topic_info_display.head(30))
print("Authoritative final run label:", FINAL_RUN_LABEL)
print(f"Saved merged-article cache to: {cache_articles_path}")
print(f"Saved merged-topic-info cache to: {cache_topic_info_path}")
print(f"Saved merged analysis metadata to: {cache_metadata_path}")
print(f"Saved rich merged-topic export to: {rich_export_path}")
print(f"Saved df_combined + Topic export to: {topic_export_path}")
if frozen_snapshot_dir is not None:
    print(f"Frozen snapshot saved to: {frozen_snapshot_dir}")
print("Rows in df_combined + Topic export:", len(thesis_topic_df))
print("Rows with final Topic assigned:", int(thesis_topic_df["Topic"].notna().sum()))
print("Rows with raw outlier Topic = -1:", int((merged_articles["merged_topic"] == -1).sum()))